In [4]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone


Create the pool table

In [5]:
conn = sqlite3.connect("screener.db", timeout=20)
cursor = conn.cursor()
conn.execute("PRAGMA journal_mode=WAL;")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pools (
    pair_address TEXT PRIMARY KEY,
    token_address TEXT,
    chain_id TEXT,
    symbol TEXT,
    dex_id TEXT,
    price_usd REAL,
    liquidity_usd REAL,
    volume_m5 REAL,
    volume_h1 REAL,
    volume_h24 REAL,
    price_change_m5 REAL,
    price_change_h1 REAL,
    price_change_h24 REAL,
    market_cap REAL,
    fdv REAL,
    pair_created_at INTEGER,
    first_seen_at TIMESTAMP,
    last_updated_at TIMESTAMP
)
""")
conn.commit()

In [6]:
response = requests.get("https://api.dexscreener.com/token-profiles/latest/v1", headers={"Accept": "*/*"})
tokens = response.json()
all_pairs = []
new_tokens_count = 0

for token in tokens:
    chain_id = token.get("chainId")
    token_address = token.get("tokenAddress")

    cursor.execute(
        """SELECT 1 FROM pools WHERE token_address = ? """,
        (token_address,)
    )

    if cursor.fetchone():
        continue

    print(f"New token discovered: {token_address}")
    new_tokens_count += 1

    now = datetime.now(timezone.utc).isoformat()
    data = requests.get(f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}",headers={"Accept": "*/*"})
    pair_response = data.json()

    pair_details = [{ "token_address" : token_address, 
                     "symbol" : item.get('baseToken').get('symbol'),
                     "pair_address" : item.get('pairAddress'),
                     "dex_id" : item.get('dexId'),
                     "price_usd" : item.get('priceUsd'),
                     "price_change_m5" : item.get('priceChange', {}).get('m5'),
                     "price_change_h1" : item.get('priceChange', {}).get('h1'),
                     "price_change_h24" : item.get('priceChange', {}).get('h24'),
                     "liquidity_usd" : item.get('liquidity', {}).get('usd'),
                     "volume_m5" : item.get('volume', {}).get('m5'),
                     "volume_h1" : item.get('volume', {}).get('h1'),
                     "volume_h24" : item.get('volume', {}).get('h24'),
                     "market_cap" : item.get('marketCap'),
                     "fdv" : item.get('fdv'),
                     "pair_created_at" : item.get('pairCreatedAt'),
                     "timestamp_fetched" : now,
    }
    for item in pair_response ]
    all_pairs.extend(pair_details)

    for pool in pair_details:
        cursor.execute(
            """
            INSERT INTO pools (
                pair_address,
                token_address,
                chain_id,
                symbol,
                dex_id,
                price_usd,
                liquidity_usd,
                volume_m5,
                volume_h1,
                volume_h24,
                price_change_m5,
                price_change_h1,
                price_change_h24,
                market_cap,
                fdv,
                pair_created_at,
                first_seen_at,
                last_updated_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(pair_address) DO UPDATE SET
                price_usd       = excluded.price_usd,
                liquidity_usd   = excluded.liquidity_usd,
                volume_m5       = excluded.volume_m5,
                volume_h1       = excluded.volume_h1,
                volume_h24      = excluded.volume_h24,
                price_change_m5  = excluded.price_change_m5,
                price_change_h1  = excluded.price_change_h1,
                price_change_h24 = excluded.price_change_h24,
                market_cap      = excluded.market_cap,
                fdv             = excluded.fdv,
                last_updated_at = excluded.last_updated_at
            """,
            (
                pool["pair_address"],
                pool["token_address"],
                chain_id,
                pool["symbol"],
                pool["dex_id"],
                pool["price_usd"],
                pool["liquidity_usd"],
                pool["volume_m5"],
                pool["volume_h1"],
                pool["volume_h24"],
                pool["price_change_m5"],
                pool["price_change_h1"],
                pool["price_change_h24"],
                pool["market_cap"],
                pool["fdv"],
                pool["pair_created_at"],
                now,
                now
            )
        )
conn.commit()

df = pd.DataFrame(all_pairs)
print(f"New tokens discovered: {new_tokens_count}")
print(f"Pairs collected: {len(df)}")
conn.close()

New token discovered: 0x7cd0D2c9ceE0f23a93aaECDD09ae17453786fb07
New token discovered: 8RzLtzrucoKMoYAFUtfu6Tcg7LZSAGar78VWREeGpump
New token discovered: axV7dLx2DuABfga7nAzCcWLJzKtEwiHbcJZLGG2pump
New token discovered: 96bPbqKyZHjRXFtmKmxmdURyL4G4hgvAfPWLjq6Xpump
New token discovered: HVv9324ouwA1VBGsynadqGdaeZKjF7iqhCHGnyKdpump
New token discovered: A4BEZe3293ZL4HKsw1smmi7uueGaKoN1ooUZafHjpump
New token discovered: Z4S3sHmJmcY44oBNFA4sJtM5Ks2dfSnniGQwi4GaX5v
New token discovered: 9Agzx1RAGUDiyURjezNJuPNsjgBcdY9nuX485h26pump
New token discovered: DTRmPLZPfQRRRVwyZFuSxUhvnj9RHgDqFjQXx6vUpump
New token discovered: HYSghNq9vCvThwCppbQaxTFP5kwjdiUu4h8d9UJppump
New token discovered: 27G8xBARqVz4k9H1jjSL7Hx4a9sHxDVvWUD1TkxZpump
New token discovered: DJ7YoBbUpTyL2gLHEBDVWpxpzSL3ANga25XrZM2Wpump
New token discovered: CzhRmEXjkuGuczWEGEkYaMoAVsK1Wrni2rhSabMmpump
New token discovered: 0xFc648b2A895a6947474bEd327c738e947c950728
New token discovered: zQV7QAY4tmiGUT5YooVV8ZjES5pLBJPVEtcqpkBpump
Ne